# SoftLayer: Adaptive Layer Skipping via Attention-Supervised Critics

**Model:** EleutherAI/pythia-1b · **Dataset:** WikiText-103 · **Framework:** PyTorch + HuggingFace

This notebook is the canonical Kaggle execution environment for the SoftLayer project.  
It pulls the latest code from the GitHub repo and runs all experiments end-to-end:

1. Environment setup & secrets
2. Repo clone + install
3. Data pipeline (WikiText-103, -100 label masking)
4. LogTemporalCritic training (attention-supervised)
5. SoftPlanningRouter skip-rate sweep
6. Baseline comparisons (static, random, token pruning, early exit)
7. Metrics: PPL · GFLOPs · Latency · CKA · Zero-shot
8. Results logged to **wandb** + local research DB

All code lives in the repo modules — this notebook is thin glue.  
To run locally: `python main.py --mode full` (see `README.md`).


## 1. Environment Setup

Reads `WANDB_API_KEY` from Kaggle Secrets (Add-ons → Secrets).  
Set `KAGGLE_DATA_PROXY_TOKEN` is handled automatically.


In [ ]:
import os, sys, subprocess

# ── Wandb secret ──────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("WANDB_API_KEY loaded from Kaggle secrets")
except Exception as e:
    print(f"[INFO] No Kaggle secrets available ({e}) — wandb will run offline")
    os.environ["WANDB_MODE"] = "offline"

# ── GPU check ─────────────────────────────────────────────────────────
import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Install Dependencies

In [ ]:
%%capture
!pip install -q transformers datasets accelerate wandb fvcore lm-eval


## 3. Clone Repository & Initialise

The repo is cloned fresh (or pulled if it exists) then added to `sys.path`  
so all local packages (`models/`, `training/`, `evaluation/`, etc.) are importable.


In [ ]:
REPO_URL = "https://github.com/Kabir08/Jumper.git"
REPO_DIR = "/kaggle/working/repo"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], check=True)
    print("Repo updated.")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


## 4. Load Config

`config.json` drives all hyperparameters: model name, dataset split, batch size,  
learning rate, target layers for the critic, wandb project name, etc.


In [ ]:
from utils.config import config
from utils.logging import setup_logging
from utils.model import load_model, device

logger = setup_logging()

print("Config loaded:")
for k, v in config.items():
    print(f"  {k:<25} = {v}")
print(f"\nDevice: {device}")


## 5. Data Pipeline

`data/dataset.py` loads WikiText-103, tokenises with the Pythia tokeniser,  
and **masks padding positions with label = -100**.  
This is critical — without masking, cross-entropy includes padding tokens  
and inflates perplexity by ~15-30 points.


In [ ]:
from torch.utils.data import DataLoader
from data.dataset import load_dataset_part

train_dataset = load_dataset_part(split=config["dataset_split"])
val_dataset   = load_dataset_part(split=config["val_dataset_split"])

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"])
val_loader   = DataLoader(val_dataset,   batch_size=config["batch_size"])

print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")

# Inspect one batch
batch = next(iter(train_loader))
print(f"input_ids shape : {batch['input_ids'].shape}")
print(f"labels shape    : {batch['labels'].shape}")
print(f"Masked positions: {(batch['labels'] == -100).sum().item()} / {batch['labels'].numel()}")


## 6. Load Base Model

Loads `EleutherAI/pythia-1b` (1.1B parameters, 16 layers, hidden_size=2048).  
The model is patched so `model.gpt_neox.layers` always exists, regardless of architecture.


In [ ]:
model, tokenizer = load_model()
print(f"Model: {config['model_name']}")
print(f"Layers: {len(model.gpt_neox.layers)}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")


## 7. Baseline: Full Model

Evaluate the unmodified model to establish the reference PPL and throughput.  
All subsequent experiments are compared against this.


In [ ]:
import wandb
from evaluation.evaluate import run_full

wandb.init(project=config.get("wandb_project", "softlayer"), name="full_model", config=config, reinit=True)

result_full = run_full(model, val_loader, device=str(device))
wandb.log(result_full)
wandb.finish()

print(f"\nFull model  PPL={result_full['ppl']:.4f}  throughput={result_full['throughput']:.1f} tok/s")


## 8. Baselines: Static & Random Skip

Quick sanity checks:
- **Static 25%**: skip the top 25% of layers unconditionally  
- **Static 50%**: skip the top 50% of layers  
- **Random 25%**: skip a random 25% (different each seed) — control group

These establish trivial compute-reduction bounds.


In [ ]:
from evaluation.evaluate import run_skip
from baselines.static_skip import apply_static_skip
from baselines.random_skip import apply_random_skip

baseline_results = []

for name, fn, kwargs in [
    ("static_25",  apply_static_skip, {"skip_rate": 0.25}),
    ("static_50",  apply_static_skip, {"skip_rate": 0.50}),
    ("random_25",  apply_random_skip, {"skip_rate": 0.25, "seed": 42}),
]:
    model, tokenizer = load_model()  # fresh unpatched model
    wandb.init(project=config.get("wandb_project", "softlayer"), name=f"baseline_{name}", config=config, reinit=True)
    r = run_skip(model, val_loader, fn, kwargs, device=str(device))
    r["method"] = name
    baseline_results.append(r)
    wandb.log(r)
    wandb.finish()
    print(f"  {name:<15}  PPL={r['ppl']:.4f}  tput={r['throughput']:.1f} tok/s")


## 9. Train LogTemporalCritic

The critic learns to predict token saliency from the attention pattern of a frozen model.

**Architecture:**
```
log1p|h| → LayerNorm(2048) → Linear(2048→256) → GELU → Linear(256→1) → Sigmoid
```

**Supervision signal:**  
For each token, the target is the column-sum of the attention weight matrix  
across `TARGET_LAYERS = [10, 12, 14]` — how much future tokens attend to this position.  
This is a proxy for "will this token matter?", computed without labels.


In [ ]:
from training.train_critic import train_block_critic
import torch

model, tokenizer = load_model()

wandb.init(project=config.get("wandb_project", "softlayer"), name="critic_train", config=config, reinit=True)
critic = train_block_critic(model, train_loader, epochs=config["epochs"], device=str(device))
wandb.finish()

torch.save(critic.state_dict(), "critic.pth")
print("Saved critic.pth")


## 10. SoftLayer: Skip-Rate Sweep

The `SoftPlanningRouter` wraps each target layer.  
At each forward pass it:
1. Runs the critic: `probs = critic(h.detach())` → per-token saliency score
2. Computes a dynamic threshold: `thresh = quantile(probs, skip_rate)`
3. Applies identity for low-saliency tokens: `h_out = layer(h)*mask + h*(1-mask)`

We sweep skip_rate ∈ {0%, 10%, 20%, 30%, 40%, 50%}.


In [ ]:
from models.critics import LogTemporalCritic
from models.router import SoftPlanningRouter
from evaluation.evaluate import evaluate_goldilocks

hidden_size = model.config.hidden_size
critic = LogTemporalCritic(in_dim=hidden_size).to(device)
critic.load_state_dict(torch.load("critic.pth", map_location=device))
critic.eval()

sweep_results = []
target_layers = config.get("target_layers", [10, 12, 14])

print(f"{'Skip %':<10} | {'PPL':<10} | {'Throughput (tok/s)':<20}")
print("-" * 45)

for skip_rate in [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]:
    model, tokenizer = load_model()

    # Wrap target layers with router
    for i in target_layers:
        base = model.gpt_neox.layers[i]
        model.gpt_neox.layers[i] = SoftPlanningRouter(base, critic, skip_rate=skip_rate)

    wandb.init(project=config.get("wandb_project", "softlayer"),
               name=f"softlayer_skip{int(skip_rate*100)}", config=config, reinit=True)

    ppl, tput = evaluate_goldilocks(model, val_loader, device=str(device))
    result = {"method": f"softlayer_{int(skip_rate*100)}pct", "skip_rate": skip_rate, "ppl": ppl, "throughput": tput}
    sweep_results.append(result)
    wandb.log(result)
    wandb.finish()

    print(f"  {int(skip_rate*100):<8}%  | {ppl:<10.4f} | {tput:<20.1f}")


## 11. Compute Efficiency: FLOPs & Latency

- **GFLOPs**: measured via `fvcore.nn.FlopCountAnalysis` and manual formula  
  `FLOPs_attn = 4Ld² + 2L²d`, `FLOPs_MLP = 8Ld²` per layer  
- **Latency**: `torch.cuda.Event` timers — 5 warm-up, 50 measured iterations  
  Reports p50, p95 in ms


In [ ]:
from evaluation.flops import measure_flops_manual
from evaluation.latency import measure_latency

model, tokenizer = load_model()

gflops = measure_flops_manual(model, seq_len=config["max_length"])
print(f"Full model GFLOPs per forward: {gflops:.2f}")

stats = measure_latency(model, seq_len=config["max_length"], device=str(device))
print(f"Latency  p50={stats.get('p50_ms', 'N/A')} ms   p95={stats.get('p95_ms', 'N/A')} ms")


## 12. Manifold Analysis: CKA per Layer

**Linear CKA** measures representational similarity between two models at each layer.  
A low CKA Δ means the skipped model preserves the same geometry as the full model — quality is intact.

$$\text{CKA}(K, L) = \frac{\text{HSIC}(K,L)}{\sqrt{\text{HSIC}(K,K) \cdot \text{HSIC}(L,L)}}$$

We compare: **full model** vs **SoftLayer at 30% skip rate**.


In [ ]:
from evaluation.manifold import layer_cka_table

model_full, _ = load_model()
model_skip, _ = load_model()

# Apply 30% skip to model_skip
for i in target_layers:
    base = model_skip.gpt_neox.layers[i]
    model_skip.gpt_neox.layers[i] = SoftPlanningRouter(base, critic, skip_rate=0.30)

cka_table = layer_cka_table(model_full, model_skip, val_loader, device=str(device))

print(f"{'Layer':<8} | {'CKA':<10} | {'CosSim':<10}")
print("-" * 35)
for row in cka_table:
    print(f"  {row['layer']:<6} | {row['cka']:<10.4f} | {row['cos_sim']:.4f}")


## 13. Zero-Shot Benchmarks (lm-eval)

Uses EleutherAI's `lm-evaluation-harness` to evaluate on:

| Task | Description |
|------|-------------|
| `hellaswag` | Commonsense NLI (sentence completion) |
| `piqa` | Physical intuition QA |
| `arc_easy` | 3rd-grade science QA (easy) |
| `arc_challenge` | Science QA (challenge) |
| `winogrande` | Winograd schema pronoun co-reference |

These test whether layer skipping degrades downstream task accuracy.


In [ ]:
from evaluation.zero_shot import run_zero_shot, print_zero_shot_table

zero_shot_results = run_zero_shot(model_name=config["model_name"])
print_zero_shot_table(zero_shot_results)


## 14. Results Summary

All results from this notebook are logged to **wandb** and to the local **research.db** (SQLite).  
You can query the DB locally with `from logs.research_logger import print_recent; print_recent()`.

### Final Comparison Table

| Method | PPL ↓ | GFLOPs ↓ | Latency ms ↓ | HellaSwag ↑ | CKA Δ ↓ |
|--------|-------|----------|--------------|-------------|---------|
| Full Model | — | — | — | — | 0.0 |
| Static 25% | *see wandb* | | | | |
| Static 50% | *see wandb* | | | | |
| Random 25% | *see wandb* | | | | |
| SoftLayer 10% | *see wandb* | | | | |
| SoftLayer 30% | *see wandb* | | | | |
| SoftLayer 50% | *see wandb* | | | | |

Fill this table after running the notebook. Results are at: **https://wandb.ai/[your-entity]/softlayer**


In [ ]:
# Print all logged events from this session
from logs.research_logger import print_recent
print("\n=== Research Log (last 30 events) ===")
print_recent(30)
